In [44]:
import numpy as np
import pandas as pd
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt

In [45]:
class Node:
    def __init__(self, feature=None, value=None, results=None, children=None):
        self.feature = feature
        self.value = value
        self.results = results
        self.children = children or {}
    

In [46]:
class ID3DecisionTree:
    def __init__(self):
        self.root = None
    def entropy(self, label):
        counts = Counter(labels)
        total = len(labels)
        return -sum((count/total)*np.log2(count/total) for count in counts)
    def information_gain(self, data, labels, feature):
        total_entropy = self.entropy(labels)
        values = np.unique(data[:, feature])
        weighted_entropy = 0
        for value in values:
            subset = labels[data[:, feature]]
            weighted_entropy += (len(subset)/len(labels)*self.entropy(subset))
        return total_entropy - weighted_entropy
    def best_feature(self, data_labels):
        features = range(data.shape[1])
        gains = [self.information_gain(data, labels, feature) for feature in features]
        return np.argmax(gains)
    def build_tree(self, data, labesls):
        if len(set(labels)) == 1:
            return Node(results=labels[0])
        
        if data.shape[1]==0:
            return Node(results=Counter(labels).most_common(1)[0][0])
        
        best = self.best_feature(data, labels)
        root = Node(feature=best)

        for value in np.unique(data[:, best]):
            indices = np.where(data[:, best]==value)
            subset_data = np.delete(data[indices], best, axis=1)
            subset_labels = labels[indices]
            root.children[value] = self.build_tree(subset_data, subset_labels)
        return root
    def fit(self, data, labels):
        self.root = self.build_tree(np.array(data), np.array(labels))
    def predict_sample(self, node, sample):
        if node.results is not None:
            return node.results
        value = sample[node.feature]
        if value in node.children:
            return self.predict_sample(node.children[value], np.delete(sample, node.feature))
        return None
    def predict(self, sample):
        return [self.predict_sample(self.root, np.array(sample)) for sample in samples]
    def visualise_tree(self, node=None, graph=None, parent_name=None, edge_label=''):
        if graph is None:
            graph = nx.DiGraph()
            node = self.root

        node_label = f'Feature {node.feature}' if node.results is None else f'Class {node.results}'
        graph.add_node(node_label)

        if parent_name is not None:
            graph.add_edge(parent_name, node_label, label=edge_label)
        for value, child in node.children.items():
            self.visualise_tree(child, graph, node_label, str(value))
        return graph
    def plot_tree(self):
        graph = self.visualise_tree()
        pos = nx.spring_layout(graph)
        labels = {node: node for node in graph.nodes}
        edge_labels =  {(u,v): d['label'] for u,v,d in graph.edges(data=True)}

        plt.figure(figsize=(10,6))
        nx.draw(graph, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=3000, font_size=10)
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8)
        plt.show()


In [47]:
# def fit(self, data, labels):
#     self.root = self.build_tree(np.array(data), np.array(labels))

In [48]:
# def predict_sample(self, node, sample):
#     if node.results is not None:
#         return node.results
#     value = sample[node.feature]
#     if value in node.children:
#         return self.predict_sample(node.children[value], np.delete(sample, node.feature))
#     return None

In [49]:
# def predict(self, sample):
#     return [self.predict_sample(self.root, np.array(sample)) for sample in samples]

In [50]:
# def visualise_tree(self, node=None, graph=None, parent_name=None, edge_label=''):
#     if graph is None:
#         graph = nx.DiGraph()
#         node = self.root

#     node_label = f'Feature {node.feature}' if node.results is None else f'Class {node.results}'
#     graph.add_node(node_label)

#     if parent_name is not None:
#         graph.add_edge(parent_name, node_label, label=edge_label)
#     for value, child in node.children.items():
#         self.visualise_tree(child, graph, node_label, str(value))
#     return graph

In [51]:
# def plot_tree(self):
#     graph = self.visualise_tree()
#     pos = nx.spring_layout(graph)
#     labels = {node: node for node in graph.nodes}
#     edge_labels =  {(u,v): d['label'] for u,v,d in graph.edges(data=True)}

#     plt.figure(figsize=(10,6))
#     nx.draw(graph, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=3000, font_size=10)
#     nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=8)
#     plt.show()

In [52]:
data = np.array([
    ['sunny', 'hot', 'high', 'weak'],
    ['sunny', 'hot', 'high', 'strong'],
    ['overcast', 'hot', 'high', 'weak'],
    ['rain', 'mild', 'high', 'weak'],
    ['rain', 'cool', 'normal', 'weak'],
    ['rain', 'cool', 'normal', 'strong'],
    ['sunny', 'mild', 'high', 'weak'],
    ['sunny', 'cool', 'cool', 'weak'],
    ['rain', 'mild', 'normal', 'weak'],
    ['sunny', 'mild', 'normal', 'strong'],
    ['overcast', 'mild', 'high', 'strong'],
    ['overcast', 'hot', 'normal', 'weak'],
    ['rain', 'mild', 'high', 'strong'],
])
labels = np.array(['No', 'No', 'Yes', 'Yes', 'Yes','No','Yes','No','Yes','Yes','Yes','Yes', 'Yes','No'])

In [53]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
for i in range(data.shape[1]):
    data[:,i] = encoder.fit_transform(data[:,i])
tree= ID3DecisionTree()

tree.fit(data, labels)

tree.plot_tree()

TypeError: ID3DecisionTree.best_feature() takes 2 positional arguments but 3 were given